# OpenLogic Finance - Logistic Regression Strategy Research Playbook (Boxes 1-3)

Welcome to the **Logistic Regression Strategy Research Playbook**. This notebook demonstrates the application of the **6-Box Architecture Model** to research, define, and backtest a machine-learning-driven trading strategy using Logistic Regression.

We follow the exact decoupled standard pattern:
- **Box 1 (Data Prep)**: Historical asset data ingestion using horizontal foundations.
- **Box 2 (Model Library)**: Zero-dependency mathematical signal and prediction logic, including walk-forward model training and weight projections between scaled feature space and raw feature space.
- **Box 3 (Strategy Testing)**: Full high-fidelity simulation using the QuantConnect LEAN Engine bridge.

---

## Box 1: Data Prep (`data_prep/`)

The **Data Prep** layer specializes in data ingestion, feature engineering, and feature storage. It is responsible for obtaining sanitised market datasets before passing them down to predictive libraries.

In this step, we will invoke the canonical data-prep module `fetch_asset_data` to load a **10-year historical dataset** for `SPY` (S&P 500 ETF), which has been robustly fetched from Yahoo Finance.

In [1]:
%load_ext autoreload
%autoreload 2

# 1. Setup path to import OpenLogic Finance packages
import sys
import os
import pandas as pd
import numpy as np

# Ensure root folder is in python path to allow clean imports
sys.path.append(os.path.abspath('../../'))

from data_prep.connectors.market_data.tools import fetch_asset_data

# Fetch a 10-year dataset for SPY (Box 1)
print("--- Box 1: Ingesting 10y Historical Data ---")
data_summary = fetch_asset_data(ticker="SPY", period="10y")
print("Data Summary:", data_summary)

# Load the generated CSV into a Pandas DataFrame
df = pd.read_csv(data_summary["csv_path"])
df["Date"] = pd.to_datetime(df["Date"], utc=True)
df.set_index("Date", inplace=True)

print(f"\nSuccessfully loaded {len(df)} rows of SPY daily data.")
df.head()

### Box 1 Ingestion: Agentic Explanation & Context Ingestion

Here, we demonstrate calling the Level 1 market data agent (`market_data_agent`) using the Google ADK/Antigravity paradigm to fetch and explain our market data for various expertise profiles.

In [2]:
# Import the Level 1 market data agent and fetch/explanation tools
from data_prep.connectors.market_data.agent import root_agent as market_data_agent
from data_prep.connectors.market_data.agent import fetch_and_explain

print("--- Box 1: Agent Definition & Capabilities ---")
print(f"Agent Name        : {market_data_agent.name}")
print(f"Agent Description : {market_data_agent.description}")
print("\n" + "="*60 + "\n")

print("--- Box 1 Ingestion: Beginner Explanation (Age 11+) ---")
beginner_res = fetch_and_explain(ticker="SPY", period="10y", explanation_level="beginner")
print(beginner_res["explanation"])
print("\n" + "="*60 + "\n")

print("--- Box 1 Ingestion: Academic Quantitative Explanation (Jim Simons Level) ---")
academic_res = fetch_and_explain(ticker="SPY", period="10y", explanation_level="academic")
print(academic_res["explanation"])

## Box 2: Model Library (`model_library/`)

The **Model Library** houses the mathematical, statistical, and indicator models. This layer is entirely decoupled from execution and testing frameworks; it represents the **pure, deterministic decision mathematics (the 'WHAT to decide')**.

Below, we perform the standard quantitative ML research pipeline:
1. **Data Slicing**: Divide the historical dataset chronologically into a 5-year **Training Set** (2016-05-31 to 2021-05-31) and a 1-year **Testing Set** (2021-06-01 to 2022-05-31) to prevent lookahead data leakage.
2. **Target Definition**: $y_t = 1$ if the `Close` price increases over the next 5 trading days, else $y_t = 0$.
3. **Feature Construction**: Calculate Wilder's RSI, Fast/Slow SMA, and momentum, then engineer features using the Box 2 `engineer_features` logic.
4. **Feature Scaling**: Fit a `StandardScaler` on the training set features and transform both the training and testing sets.
5. **Model Training**: Fit a `LogisticRegression` model on the training set.
6. **Out-of-Sample Testing**: Generate predictions and evaluate validation metrics (Accuracy, Precision, Recall, F1-Score, and ROC-AUC).

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from model_library.ml_zoo.logistic_regression import engineer_features

# 1. Define Wilder's RSI function in Pandas
def pandas_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period-1, adjust=False).mean()
    avg_loss = loss.ewm(com=period-1, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# 2. Build indicators matching Main.py settings
df_ml = df.copy()
df_ml['Fast_SMA'] = df_ml['Close'].rolling(window=50).mean()
df_ml['Slow_SMA'] = df_ml['Close'].rolling(window=200).mean()
df_ml['RSI'] = pandas_rsi(df_ml['Close'], 14)
df_ml['Prev_Close'] = df_ml['Close'].shift(1)

# 3. Apply Box 2 feature engineering
features_list = []
for idx, row in df_ml.iterrows():
    raw_item = {
        "close": row['Close'],
        "fast_sma": row['Fast_SMA'],
        "slow_sma": row['Slow_SMA'],
        "rsi": row['RSI'],
        "prev_close": row['Prev_Close']
    }
    features_list.append(engineer_features(raw_item))

feat_df = pd.DataFrame(features_list, index=df_ml.index)
feat_df['Target'] = (df_ml['Close'].shift(-5) > df_ml['Close']).astype(int)

# Drop first 200 rows (due to Slow SMA lookback) and last 5 rows (due to Target shift)
clean_df = feat_df.dropna().copy()

# 4. Split chronologically into Train (5 years) and Test (1 year)
train_df = clean_df.loc[:'2021-05-31']
test_df = clean_df.loc['2021-06-01':'2022-05-31']

feature_names = ['sma_ratio', 'rsi_norm', 'momentum']
X_train = train_df[feature_names]
y_train = train_df['Target']

X_test = test_df[feature_names]
y_test = test_df['Target']

# 5. Feature Scaling (fit on train, transform on both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train model
lr_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# 7. Generate predictions on Testing Set
y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# 8. Calculate and print Out-of-Sample metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("--- Walk-Forward Out-of-Sample Performance Metrics (1-Year Test) ---")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")

# 9. Extract and print coefficients for Deployment
weights = {name: coef for name, coef in zip(feature_names, lr_model.coef_[0])}
intercept = float(lr_model.intercept_[0])
feature_means = {name: mean for name, mean in zip(feature_names, scaler.mean_)}
feature_stds = {name: std for name, std in zip(feature_names, scaler.scale_)}

print("\n--- Derived LogisticModelPayload Parameters (Ready for Box 3 Deployment) ---")
print(f"weights = {weights}")
print(f"intercept = {intercept:.8f}")
print(f"feature_means = {feature_means}")
print(f"feature_stds = {feature_stds}")

### Box 2 Deployment Payload & Verification

Now we load our dynamically derived weights and intercept into the standard `LogisticModelPayload` to verify predictions and perform raw projected weights validation.

In [4]:
from model_library.ml_zoo.logistic_regression import (
    LogisticStrategyConfig,
    LogisticModelPayload,
    engineer_features,
    predict_probability,
    project_weights
)

# 1. Initialize our strategy config and pretrained model payload
config = LogisticStrategyConfig(
    ticker="SPY",
    fast_period=50,
    slow_period=200,
    rsi_period=14,
    probability_threshold=0.5,
    position_size=1.0,
    max_drawdown_pct=0.15
)

# Use the dynamically trained parameters derived above
model_payload = LogisticModelPayload(
    weights=weights,
    intercept=intercept,
    feature_means=feature_means,
    feature_stds=feature_stds
)

print("Model weights derived from walk-forward training:")
for f, w in model_payload.weights.items():
    print(f"  - {f}: {w:.4f}")
print(f"  - intercept (bias): {model_payload.intercept:.4f}")

### Box 2 Verification: Raw Weight Projection Equivalence

One of the vital mathematical aspects of deploying standardized machine learning models in high-frequency or daily execution systems is efficiency. 

Rather than standardizing features on every tick using means and standard deviations, we can project the weights back to the **raw feature space**. Below, we demonstrate our Box 2 weight projection helper and prove that computing probabilities using raw feature space yields identical results to scaled feature space calculation.

In [5]:
raw_weights, raw_intercept = project_weights(model_payload)
print("Projected Weights (Raw Space):")
for f, w_raw in raw_weights.items():
    print(f"  - {f}: {w_raw:.4f}")
print(f"  - raw intercept: {raw_intercept:.4f}")

print("\n--- Mathematically proving raw space projection equivalence ---")

# Create sample raw market data
sample_raw = {
    "close": 105.0,
    "fast_sma": 102.0,
    "slow_sma": 100.0,
    "rsi": 60.0,
    "prev_close": 100.0,
}

# Engineer features
feats = engineer_features(sample_raw)
print(f"Engineered raw features: {feats}")

# 1. Calculate probability using scaled features method
prob_scaled = predict_probability(feats, model_payload)

# 2. Calculate probability directly using raw projected weights
z_raw = raw_intercept + sum(raw_weights[f] * feats[f] for f in feats)
prob_raw = 1.0 / (1.0 + np.exp(-z_raw))

print(f"  -> Probability (scaled method) : {prob_scaled:.8f}")
print(f"  -> Probability (raw projected) : {prob_raw:.8f}")
print(f"  -> Mathematical Equivalence   : {abs(prob_scaled - prob_raw) < 1e-12}")

## Box 3: Strategy Testing (`strategy_testing/`)

The **Strategy Testing** layer is responsible for assessing performance and estimating risk under simulation.

To test this model strategy against high-fidelity daily stock data, we interface with the QuantConnect LEAN engine using the repository's **`LeanEngineBridge`**.

We will run the bridge, which automatically syncs our Box 2 Logistic Regression module into the local QuantConnect workspace, patches parameters, and runs a premium high-fidelity cloud backtest on QuantConnect Cloud.

In [6]:
from strategy_testing.lean_engine import LeanEngineBridge

print("--- Box 3: Strategy Testing via QuantConnect LEAN Engine ---")

# Initialize the bridge adapter
bridge = LeanEngineBridge()
lean_check = bridge.check_lean_installed()
print(f"LEAN CLI Installed: {lean_check['installed']}")

if lean_check['installed']:
    print(f"LEAN CLI Version: {lean_check['version']}")
    
    # Run the backtest for SPY with our model parameters
    print("\nRunning LEAN High-Fidelity Backtest... (Syncing and pushing ML signals)")
    res = bridge.run_backtest(
        ticker='SPY',
        fast_period=50,
        slow_period=200,
        rsi_period=14,
        probability_threshold=0.5,
        max_drawdown_pct=0.99  # Disable drawdown risk stop for standard comparison
    )
    
    if res.success:
        print("\n✅ LEAN BACKTEST COMPLETED SUCCESSFULLY!")
        
        if res.full_summary:
            print("\n=== QuantConnect LEAN Backtest Results Summary ===")
            print(res.full_summary)
    else:
        print(f"\n❌ LEAN Backtest Failed: {res.stderr}")
else:
    print("\nNote: QuantConnect LEAN CLI not installed locally. Run 'pip install lean' and check environment configuration to enable high-fidelity backtests.")